# 🩺 Syght — Healthcare Insurance Fraud Detector Platform
## Unit-1 Experiment 1: Introduction to Data Pre-processing, Pandas, NumPy & Matplotlib
**Course Specialization**: AI & Machine Learning with Cloud & Azure Tools  
**Project**: Healthcare Insurance Fraud Detection ()  

---

### 🎯 Experiment Aim & Overview:
Demonstrates foundational data cleaning, missing value handling, date parsing, NumPy vectorized transforms, and Matplotlib data visualization on healthcare insurance claims.

---

### 📚 Theoretical Background
In healthcare insurance billing, data is split across multiple relational tables:
1. **Inpatient Data**: Overnight hospitalizations, admission dates, discharge dates, Diagnosis-Related Groups (DRG).
2. **Outpatient Data**: Day clinic visits, ambulatory diagnostic procedures, attending physicians.
3. **Beneficiary Data**: Patient demographic and clinical history (11 chronic disease indicators).
4. **Provider Labels**: Audit determinations (PotentialFraud = 'Yes'/'No').

**Key Challenges**:
- High missingness in optional fields (Operating Physicians, Deductibles).
- Extreme skewness in financial reimbursement distributions.
- String dates requiring conversion for Length of Stay (LOS) calculations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visual styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
print('NumPy, Pandas, and Matplotlib imported successfully!')

### Step 1: Ingestion of Multi-Table Insurance Claims

In [ ]:
# Load synthetic or raw claims data
from ml_pipeline.medicare_simulator import generate_simulated_medicare_data

labels, bene, inp, outp = generate_simulated_medicare_data(n_providers=50, seed=42)
print(f'Provider Labels Shape: {labels.shape}')
print(f'Beneficiary Records Shape: {bene.shape}')
print(f'Inpatient Admissions Shape: {inp.shape}')
print(f'Outpatient Visits Shape: {outp.shape}')

inp.head(3)

### Step 2: Missing Value Audit & Preprocessing with Pandas

In [ ]:
# Check structural nulls
null_summary = pd.DataFrame({
    'Inpatient_Nulls': inp.isnull().sum(),
    'Inpatient_Null_Pct': (inp.isnull().sum() / len(inp)) * 100
})
print('Missing Values in Inpatient Admissions:')
print(null_summary[null_summary['Inpatient_Nulls'] > 0])

# Impute deductibles and handle categorical missing values
inp_cleaned = inp.copy()
inp_cleaned['DeductibleAmtPaid'] = inp_cleaned['DeductibleAmtPaid'].fillna(0.0)
inp_cleaned['OperatingPhysician'] = inp_cleaned['OperatingPhysician'].fillna('NONE')
print(f'Remaining nulls in DeductibleAmtPaid: {inp_cleaned["DeductibleAmtPaid"].isnull().sum()}')

### Step 3: Date Transformation & Feature Derivation with NumPy

In [ ]:
# Convert string dates to datetime and compute Length of Stay (LOS)
inp_cleaned['AdmissionDt'] = pd.to_datetime(inp_cleaned['AdmissionDt'])
inp_cleaned['DischargeDt'] = pd.to_datetime(inp_cleaned['DischargeDt'])
inp_cleaned['LengthOfStay'] = (inp_cleaned['DischargeDt'] - inp_cleaned['AdmissionDt']).dt.days

# NumPy log-transformation to stabilize heavy-tailed financial amounts
inp_cleaned['Log_Reimbursed'] = np.log1p(np.maximum(inp_cleaned['InscClaimAmtReimbursed'].to_numpy(), 0.0))

inp_cleaned[['LengthOfStay', 'InscClaimAmtReimbursed', 'Log_Reimbursed']].describe()

### Step 4: Matplotlib Visual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram of raw reimbursement
axes[0].hist(inp_cleaned['InscClaimAmtReimbursed'], bins=25, color='#3b82f6', edgecolor='black', alpha=0.7)
axes[0].set_title('Raw Reimbursement Amount Distribution (Highly Skewed)')
axes[0].set_xlabel('Amount (₹ / $)')
axes[0].set_ylabel('Claim Frequency')

# 2. Histogram of log-transformed reimbursement
axes[1].hist(inp_cleaned['Log_Reimbursed'], bins=25, color='#10b981', edgecolor='black', alpha=0.7)
axes[1].set_title('NumPy Log-Transformed Distribution (Bell-Shaped / Gaussian)')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Claim Frequency')

plt.tight_layout()
plt.show()